# Question 9 - Country Exposure Map

In [ ]:
from pathlib import Path
import pickle
import pycountry
import matplotlib.pyplot as plt

with Path("Data/question7_countries_by_birth_year.pkl").open("rb") as file:
    countries = pickle.load(file)
with Path("Data/gdf_country_borders.pkl").open("rb") as file:
    borders = pickle.load(file).rename(columns={"abbreviation": "iso3"})

aliases = {
    "Bolivia (Plurinational State of)": "BOL",
    "Democratic Republic of the Congo": "COD",
    "China, Hong Kong SAR": "HKG",
    "Iran (Islamic Republic of)": "IRN",
    "Dem. People's Republic of Korea": "PRK",
    "Republic of Korea": "KOR",
    "Kosovo (under UNSC res. 1244)": "XKX",
    "China, Taiwan Province of China": "TWN",
    "Venezuela (Bolivarian Republic of)": "VEN",
    "United States Virgin Islands": "VIR",
    "State of Palestine": "PSE",
}
hazards = {
    "burntarea": "Wildfire",
    "cropfailedarea": "Crop failure",
    "driedarea": "Drought",
    "floodedarea": "River flood",
    "heatwavedarea": "Heatwave",
    "tropicalcyclonedarea": "Tropical cyclone",
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
for ax, hazard in zip(axes.flat, countries["extreme"].values):
    medians = countries["median_val"].sel(birth_years=2025, extreme=hazard).squeeze()
    values = {}
    for name in medians["country"].values:
        if name in aliases:
            code = aliases[name]
        else:
            code = pycountry.countries.lookup(name).alpha_3
        values[code] = float(medians.sel(country=name))

    map_data = borders.copy()
    map_data["median"] = map_data["iso3"].map(values)
    map_data.plot(column="median", cmap="RdYlBu_r", legend=True, ax=ax,
                  edgecolor="white", linewidth=0.2,
                  missing_kwds={"color": "lightgrey"})
    ax.set_title(hazards[hazard])
    ax.set_axis_off()

fig.suptitle("Additional exposure for people born in 2025")
output = Path("figures/question9_country_exposure_2025.png")
output.parent.mkdir(exist_ok=True)
fig.savefig(output, dpi=300, bbox_inches="tight")
display(fig)